# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Input offline** nằm riêng theo notebook: `input/agoda-thy-3.csv` (cell ① `OFFLINE_FILE` / cell tải Sheet).

**Output** nằm trong `results/agoda/<RUN_NAME>/` — notebook này dùng `thy-3`, không ghi đè notebook khác:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

Cache warm (`results/agoda/captures/`) vẫn dùng chung giữa các notebook nên không tốn thêm thời gian warm.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda/<RUN_NAME>/ ──
RUN_NAME = "thy-3"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1vttGK0gOuTnA5shkfFxOlm1R9xyEEmKxAuDxhhTInnY/edit?gid=1083140588#gid=1083140588"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/agoda-thy-3.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 23 khách sạn. 5 dòng đầu:
   • Vanda Hotel — Superior Double City View
   • G8 Luxury — Superior Double Room
   • Hilton Da Nang — King Accessible Room
   • Crowne Plaza Danang City Centre — Standard Room
   • Novotel — Superior Room, balcony with Han river view - 2 single beds


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda-thy-3.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)   # mỗi notebook 1 thư mục kết quả riêng
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
    # cache warm dùng CHUNG cho mọi notebook (đỡ warm lại) — chỉ kết quả là tách riêng
    capture_dir=os.path.join(ROOT, "results", "agoda", "captures"),
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 23 rows from TEMP_agoda.csv
🚀 AGODA crawl | 23 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-09-08
🦊 Camoufox ready (humanize=True geoip=True) — browser navs use anti-detect Firefox
✔️  1/23 Vanda Hotel — complete, skip
✔️  2/23 G8 Luxury — complete, skip
✔️  3/23 Hilton Da Nang — complete, skip
✔️  4/23 Crowne Plaza Danang City Centre — complete, skip
✔️  5/23 Novotel — complete, skip
✔️  6/23 M Hotel Da Nang — complete, skip
✔️  7/23 Muong Thanh Luxury Song Han Hotel — complete, skip
✔️  8/23 New Orient Da Nang — complete, skip
✔️  9/23 radison red — complete, skip
✔️  10/23 sel de mer — complete, skip
✔️  11/23 A La Carte — complete, skip
✔️  12/23 voco Ma Belle — complete, skip
✔️  13/23 Grand Tourane Hotel — complete, skip
✔️  14/23 Hilton Garden Inn — complete, skip
✔️  15/23 Pullman Danang Beach Resort — complete, skip
✔️  16/23 Furama Villas Danang-1 Bedroom Pool Villa — complete, skip
✔️  17/23 Furama Villas Danang-2 Bedroom Pool Villa — complete, skip
✔️

'FINAL_20260903.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/thy-3/FINAL_20260903.csv — 23 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,Vanda Hotel,Superior Double City View,"1,062,828","1,078,091","1,188,914","1,088,126","1,337,356","1,104,302"
1,G8 Luxury,Superior Double Room,"644,791","654,715","654,715","659,593","650,192","648,080"
2,Hilton Da Nang,King Accessible Room,"2,262,277","2,500,000","2,262,277","2,262,277","2,300,000","2,620,000"
3,Crowne Plaza Danang City Centre,Standard Room,"1,981,747","1,981,747","1,981,747","1,747,066","1,747,066","1,668,840"
4,Novotel,"Superior Room, balcony with Han river view - 2...","1,900,000","1,900,000","1,900,000","1,750,000","1,750,000","1,700,000"
5,M Hotel Da Nang,Phòng Loại Sang Hai Giường Đơn Hướng Thành Phố...,"2,369,839","3,026,455","3,026,455","3,026,455","2,910,053","2,865,961"
6,Muong Thanh Luxury Song Han Hotel,Superior Twin City View,"851,365","795,472","795,472","795,472","795,472","795,472"
7,New Orient Da Nang,Superior Double or Twin Room without View,"1,259,921","990,439","990,439","990,439","990,439","1,204,365"
8,radison red,Superior Room,"2,365,059","1,935,000","1,935,000","1,935,000","2,365,059","1,665,000"
9,sel de mer,Superior Twin City View,"2,045,855","2,186,949","2,045,855","1,975,309","1,975,309","1,763,668"
